# T31 机构行为监测 - 重构 Notebook

## 项目概述

机构行为监测项目用于监测和分析债券市场机构投资者的交易行为。

### 主要功能
- 机构交易数据采集
- 行为模式分析
- 多维度可视化监测
- 风险预警

### 14个监测图表

| 图表 | 内容 | 类型 |
|------|------|------|
| 图表1 | 各机构类型每日净交易量 | 柱状图 |
| 图表2 | 机构间交易矩阵 | 热力图 |
| 图表3 | 机构期限偏好分析 | 树形图 |
| 图表4 | 机构券种偏好分析 | 树形图 |
| 图表5 | 市场份额时序分析 | 折线图 |
| 图表6 | 机构交易热点分析 | 柱状图 |
| 图表7 | 全国城投利差热力图 | 地图热力图 |
| 图表8 | 市场核心行为指数 | 多轴折线图 |
| 图表9 | 市场风险定价 | 双轴折线图 |
| 图表10 | 机构行为记分卡 | 记分卡 |
| 图表11 | 资金面与杠杆温度计 | 温度计 |
| 图表12 | 机构大类券种配置流向 | 瀑布图 |
| 图表13 | 十大活跃券追踪 | 排行榜 |
| 图表14 | 区域城投债资金来源 | 桑基图 |

---

## 1. 环境配置

### 1.1 导入依赖包

In [ ]:
# 标准库
import os
import sys
import logging
from datetime import datetime, timedelta
from typing import Optional, Dict, Any, List, Tuple

# 数据处理
import pandas as pd
import numpy as np

# 可视化
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# 数据库
from sqlalchemy import create_engine, text
from sqlalchemy.pool import NullPool

# 环境变量
from dotenv import load_dotenv

# 本地模块
from config import (
    DatabaseConfig, ChartConfig, INSTITUTION_TYPES, TENOR_CATEGORIES,
    BOND_ASSET_CLASSES, PROVINCES, CHART_CONFIGS,
    map_tenor_to_category, map_bond_type_to_asset_class
)
from utils import (
    DatabaseManager, clean_numeric_column, clean_date_column,
    get_valid_date_range, setup_chinese_font, format_number
)
from charts import InstitutionBehaviorMonitor

print("依赖包导入完成!")

### 1.2 配置参数

In [ ]:
# 加载环境变量
load_dotenv()

# 设置中文字体
setup_chinese_font()

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# 显示设置
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.2f}'.format)

# 日期参数
end_date = datetime.now().strftime('%Y-%m-%d')
start_date_30d = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')
start_date_90d = (datetime.now() - timedelta(days=90)).strftime('%Y-%m-%d')
start_date_1y = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')

print(f"当前日期: {end_date}")
print(f"30天前: {start_date_30d}")
print(f"90天前: {start_date_90d}")
print(f"1年前: {start_date_1y}")

### 1.3 数据库连接

In [ ]:
# 从环境变量获取数据库配置
db_config = DatabaseConfig(database='bond')

# 创建数据库管理器
db_manager = DatabaseManager(db_config.get_connection_url())

# 测试连接
if db_manager.test_connection():
    print("数据库连接成功!")
else:
    print("数据库连接失败，请检查环境变量配置")
    print("需要设置: DB_USER, DB_PASSWORD, DB_HOST, DB_NAME")

---
## 2. 数据获取

### 2.1 数据同步（可选）

如果需要从Excel文件同步数据到数据库，可以使用以下代码。

In [ ]:
# 数据同步示例（取消注释以执行）
# from data_sync import sync_institution_data
#
# source_dir = "/data/项目/快速处理/2025/机构行为/"
# results = sync_institution_data(source_dir, db_manager)
# print(f"同步结果: {results}")

### 2.2 查看数据表结构

In [ ]:
# 查看主表结构
table_info = db_manager.get_table_info('现券成交分机构统计表')
print("=== 现券成交分机构统计表 结构 ===")
print(table_info)

In [ ]:
# 查看数据样例
sample_sql = """
SELECT * FROM bond.`现券成交分机构统计表` 
ORDER BY `交易日期` DESC 
LIMIT 10
"""
sample_df = db_manager.execute_query(sample_sql)
print("=== 数据样例 ===")
print(sample_df)

---
## 3. 数据处理

### 3.1 创建监测器实例

In [ ]:
# 创建监测器
monitor = InstitutionBehaviorMonitor(db_manager)

print("监测器创建成功!")

---
## 4. 核心逻辑 - 14个监测图表

### 图表1: 各机构类型每日净交易量

In [ ]:
# 获取数据
df_chart1 = monitor.get_chart1_daily_net_trading(date=end_date)

print("=== 图表1: 各机构类型每日净交易量 ===")
print(df_chart1)

# 可视化
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['green' if x > 0 else 'red' for x in df_chart1['value']]
df_chart1.plot(kind='bar', x='name', y='value', ax=ax, color=colors, legend=False)
ax.set_title(f'各机构类型净交易量 ({end_date})')
ax.set_xlabel('机构类型')
ax.set_ylabel('净买入量（亿元）')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 图表2: 机构间交易矩阵

In [ ]:
# 获取数据
chart2_data = monitor.get_chart2_trading_matrix(date=end_date)

print("=== 图表2: 机构间交易矩阵 ===")
print(f"买方类型: {chart2_data['categories_table'][0]['v']}")
print(f"卖方类型: {chart2_data['categories_table'][1]['v']}")
print(f"数据点数量: {len(chart2_data['final'])}")

### 图表3: 机构期限偏好分析

In [ ]:
# 获取数据
institution_type = '大型商业银行/政策性银行'
df_chart3 = monitor.get_chart3_tenor_preference(date=end_date, institution_type=institution_type)

print(f"=== 图表3: {institution_type} 期限偏好 ===")
print(df_chart3)

# 可视化
if not df_chart3.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    df_chart3.plot(kind='pie', y='value', labels=df_chart3['name'], ax=ax, autopct='%1.1f%%')
    ax.set_title(f'{institution_type} 期限偏好分析 ({end_date})')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()

### 图表4: 机构券种偏好分析

In [ ]:
# 获取数据
df_chart4 = monitor.get_chart4_bond_type_preference(date=end_date, institution_type=institution_type)

print(f"=== 图表4: {institution_type} 券种偏好 ===")
print(df_chart4)

# 可视化
if not df_chart4.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    df_chart4.plot(kind='bar', x='name', y='value', ax=ax, legend=False)
    ax.set_title(f'{institution_type} 券种偏好分析 ({end_date})')
    ax.set_xlabel('券种类型')
    ax.set_ylabel('交易量（亿元）')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

### 图表5: 市场份额时序分析

In [ ]:
# 获取数据
df_chart5 = monitor.get_chart5_market_share_timeseries(start_date=start_date_30d, end_date=end_date)

print("=== 图表5: 市场份额时序分析 ===")
print(df_chart5.head())

# 可视化
if not df_chart5.empty:
    fig, ax = plt.subplots(figsize=(14, 7))
    df_chart5.plot(kind='line', ax=ax, marker='o', markersize=3)
    ax.set_title('各机构市场份额时序变化')
    ax.set_xlabel('日期')
    ax.set_ylabel('市场份额 (%)')
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

### 图表6: 机构交易热点分析

In [ ]:
# 获取数据
df_chart6 = monitor.get_chart6_trading_hotspots(date=end_date, institution_type=institution_type, top_n=10)

print(f"=== 图表6: {institution_type} 交易热点 ===")
print(df_chart6[['name', 'total_amount', 'net_buy']])

# 可视化
if not df_chart6.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    colors = ['green' if x > 0 else 'red' for x in df_chart6['net_buy']]
    df_chart6.plot(kind='barh', x='name', y='total_amount', ax=ax, color=colors, legend=False)
    ax.set_title(f'{institution_type} 交易热点 TOP10')
    ax.set_xlabel('交易金额（亿元）')
    ax.set_ylabel('债券名称')
    plt.tight_layout()
    plt.show()

### 图表7: 全国城投利差热力图

In [ ]:
# 获取数据
df_chart7 = monitor.get_chart7_city_invest_spread_heatmap(date=end_date)

print("=== 图表7: 全国城投利差热力图 ===")
print(df_chart7.sort_values('value', ascending=False).head(10))

# 可视化
if not df_chart7.empty:
    fig, ax = plt.subplots(figsize=(14, 8))
    df_sorted = df_chart7.sort_values('value', ascending=False).head(20)
    colors = plt.cm.RdYlGn_r(np.linspace(0, 1, len(df_sorted)))
    df_sorted.plot(kind='bar', x='name', y='value', ax=ax, color=colors, legend=False)
    ax.set_title(f'全国城投利差 TOP20 ({end_date})')
    ax.set_xlabel('省份')
    ax.set_ylabel('利差 (BP)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

### 图表8: 市场核心行为指数

In [ ]:
# 获取数据
df_chart8 = monitor.get_chart8_core_behavior_index(start_date=start_date_90d, end_date=end_date)

print("=== 图表8: 市场核心行为指数 ===")
print(df_chart8.head())

# 可视化
if not df_chart8.empty:
    fig, ax = plt.subplots(figsize=(14, 6))
    df_chart8.plot(kind='line', x='date', y='rfi_premium', ax=ax, marker='o', markersize=3)
    ax.set_title('RFY Premium 指数')
    ax.set_xlabel('日期')
    ax.set_ylabel('RFY Premium')
    ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

### 图表9: 市场风险定价

In [ ]:
# 获取数据
df_chart9 = monitor.get_chart9_risk_pricing(start_date=start_date_1y, end_date=end_date)

print("=== 图表9: 市场风险定价 ===")
print(df_chart9.head())

# 可视化
if not df_chart9.empty:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

    if 'term_spread' in df_chart9.columns:
        df_chart9.plot(kind='line', x='date', y='term_spread', ax=ax1, marker='o', markersize=2)
        ax1.set_title('期限利差 (10Y-1Y)')
        ax1.set_ylabel('利差 (%)')

    if 'credit_spread' in df_chart9.columns:
        df_chart9.plot(kind='line', x='date', y='credit_spread', ax=ax2, marker='o', markersize=2, color='orange')
        ax2.set_title('信用利差 (AA+ 3Y中票 - 3Y国债)')
        ax2.set_ylabel('利差 (BP)')

    for ax in [ax1, ax2]:
        ax.set_xlabel('日期')
        ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

    plt.tight_layout()
    plt.show()

### 图表10: 机构行为记分卡

In [ ]:
# 获取数据
institution_type = '基金公司及产品'
df_chart10 = monitor.get_chart10_behavior_scorecard(date=end_date, institution_type=institution_type)

print(f"=== 图表10: {institution_type} 行为记分卡 ===")
print(df_chart10)

# 可视化
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

# 创建表格
table_data = [[row['indicator'], f"{row['value']:.4f}", f"{row['change']:+.4f}"] 
              for _, row in df_chart10.iterrows()]
table = ax.table(cellText=table_data, 
                 colLabels=['指标', '当前值', '变化'],
                 loc='center', 
                 cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.5)

ax.set_title(f'{institution_type} 行为记分卡 ({end_date})', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

### 图表11: 资金面与杠杆温度计

In [ ]:
# 获取数据
chart11_data = monitor.get_chart11_fund_leverage_thermometer(date=end_date)

print("=== 图表11: 资金面与杠杆温度计 ===")
for key, value in chart11_data.items():
    print(f"{key}: {value}")

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 温度计样式显示
metrics = [
    ('回购余额(亿)', chart11_data.get('total_repo_balance', 0), 'blue'),
    ('DR007(%)', chart11_data.get('dr007', 0), 'green'),
    ('R007(%)', chart11_data.get('r007', 0), 'red')
]

for ax, (name, value, color) in zip(axes, metrics):
    ax.bar([0], [value], color=color, width=0.5)
    ax.set_xlim(-0.5, 0.5)
    ax.set_xticks([0])
    ax.set_xticklabels([name])
    ax.set_title(f'{value:.2f}' if value else 'N/A')

plt.suptitle(f'资金面温度计 ({chart11_data.get("date", end_date)})', fontsize=14)
plt.tight_layout()
plt.show()

### 图表12: 机构大类券种配置流向

In [ ]:
# 获取数据
institution_type = '基金公司及产品'
df_chart12 = monitor.get_chart12_asset_flow(start_date=start_date_30d, end_date=end_date, institution_type=institution_type)

print(f"=== 图表12: {institution_type} 大类券种配置流向 ===")
print(df_chart12)

# 可视化（瀑布图）
if not df_chart12.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['green' if x > 0 else 'red' for x in df_chart12['value']]
    df_chart12.plot(kind='bar', x='name', y='value', ax=ax, color=colors, legend=False)
    ax.set_title(f'{institution_type} 大类资产配置流向 (近30天)')
    ax.set_xlabel('资产类型')
    ax.set_ylabel('净流入（亿元）')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

### 图表13: 十大活跃券追踪

In [ ]:
# 获取数据
df_chart13 = monitor.get_chart13_top_active_bonds(start_date=start_date_30d, end_date=end_date, top_n=10)

print("=== 图表13: 十大活跃券追踪 ===")
print(df_chart13[['name', 'total_amount', 'trade_count']])

# 可视化
if not df_chart13.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    df_chart13.plot(kind='barh', x='name', y='total_amount', ax=ax, legend=False, color='steelblue')
    ax.set_title('十大活跃券 (近7天)')
    ax.set_xlabel('成交金额（亿元）')
    ax.set_ylabel('债券名称')
    ax.invert_yaxis()  # 让第一名在最上面
    plt.tight_layout()
    plt.show()

### 图表14: 区域城投债资金来源

In [ ]:
# 获取数据
province = '江苏'
df_chart14 = monitor.get_chart14_city_invest_fund_source(start_date=start_date_90d, end_date=end_date, province=province)

print(f"=== 图表14: {province}城投债资金来源 ===")
print(df_chart14[['source', 'value']])

# 可视化
if not df_chart14.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    df_chart14.plot(kind='pie', y='value', labels=df_chart14['source'], ax=ax, autopct='%1.1f%%')
    ax.set_title(f'{province}城投债资金来源 (近90天)')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()

---
## 5. 执行与测试

### 5.1 批量生成所有图表

In [ ]:
def generate_all_charts(monitor, end_date, start_date_30d, start_date_90d, start_date_1y):
    """批量生成所有图表数据"""
    results = {}
    
    try:
        results['chart1'] = monitor.get_chart1_daily_net_trading(date=end_date)
        print("[OK] 图表1 - 每日净交易量")
    except Exception as e:
        print(f"[FAIL] 图表1: {e}")
        results['chart1'] = None

    try:
        results['chart3'] = monitor.get_chart3_tenor_preference(date=end_date)
        print("[OK] 图表3 - 期限偏好")
    except Exception as e:
        print(f"[FAIL] 图表3: {e}")
        results['chart3'] = None

    try:
        results['chart4'] = monitor.get_chart4_bond_type_preference(date=end_date)
        print("[OK] 图表4 - 券种偏好")
    except Exception as e:
        print(f"[FAIL] 图表4: {e}")
        results['chart4'] = None

    try:
        results['chart5'] = monitor.get_chart5_market_share_timeseries(start_date=start_date_30d, end_date=end_date)
        print("[OK] 图表5 - 市场份额时序")
    except Exception as e:
        print(f"[FAIL] 图表5: {e}")
        results['chart5'] = None

    try:
        results['chart12'] = monitor.get_chart12_asset_flow(start_date=start_date_30d, end_date=end_date)
        print("[OK] 图表12 - 大类资产流向")
    except Exception as e:
        print(f"[FAIL] 图表12: {e}")
        results['chart12'] = None

    try:
        results['chart13'] = monitor.get_chart13_top_active_bonds(start_date=start_date_30d, end_date=end_date)
        print("[OK] 图表13 - 十大活跃券")
    except Exception as e:
        print(f"[FAIL] 图表13: {e}")
        results['chart13'] = None

    return results

# 执行批量生成
print("=== 批量生成所有图表 ===")
all_charts = generate_all_charts(monitor, end_date, start_date_30d, start_date_90d, start_date_1y)
print(f"\n成功生成 {sum(1 for v in all_charts.values() if v is not None)} 个图表")

### 5.2 数据导出

In [ ]:
import json

# 导出数据为JSON
output_dir = './output'
os.makedirs(output_dir, exist_ok=True)

for chart_name, df in all_charts.items():
    if df is not None and isinstance(df, pd.DataFrame):
        output_path = os.path.join(output_dir, f'{chart_name}.json')
        df.to_json(output_path, orient='records', force_ascii=False, indent=2)
        print(f"导出: {output_path}")

print(f"\n所有数据已导出到 {output_dir}")

---
## 6. 工具函数

### 6.1 数据库连接关闭

In [ ]:
# 关闭数据库连接
db_manager.close()
print("数据库连接已关闭")

---
## 总结

本Notebook完成了机构行为监测项目的重构，包含以下内容：

1. **项目概述** - 14个监测图表的功能说明
2. **环境配置** - 依赖导入、参数配置、数据库连接
3. **数据获取** - 数据同步、表结构查看
4. **数据处理** - 监测器实例创建
5. **核心逻辑** - 14个图表的数据处理和可视化
6. **执行与测试** - 批量生成和数据导出

### 文件清单

| 文件 | 说明 |
|------|------|
| config.py | 配置模块（数据库、图表、机构类型等） |
| utils.py | 工具函数（数据库操作、数据清洗等） |
| charts.py | 核心逻辑（14个图表生成函数） |
| data_sync.py | 数据同步模块 |
| requirements.txt | 依赖包列表 |
| .env.example | 环境变量示例 |

---